# 06 GLASS Interactive SI Shift

Display interactive latest-observed-count vs future SI-shift plots, with GLASS pathogen-antibiotic pairs highlighted.

## Inputs

- Screened yearly pairs from notebook 04c.
- Yearly SI and eligible-pair outputs from notebook 05.
- GLASS pair data and the configured pathogen/antibiotic lookup tables.

## Outputs

- Interactive GLASS comparison figures, PDF exports, and supporting CSV tables under the configured `outputs/figures/` directory.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipelines import (
    build_glass_match_table,
    build_novel_prediction_table,
    build_si_analysis_paths,
    load_or_build_interactive_rank,
    load_si_and_clean_glass_list,
    plot_future_si_vs_si_shift,
    plot_latest_si_vs_future_si,
    summarize_pathogen_coverage,
)
from amr.si import plot_interactive_latest_observed_count_vs_future_si_shift
from amr.pipeline_config import cfg, cfg_path, load_pipeline_config


## Parameters

In [3]:
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)

OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.output_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.parquet_dir", PROJECT_ROOT)
FIGURE_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.figure_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR
ASSETS_DIR = cfg_path(PIPELINE_CONFIG, "paths.assets_dir", PROJECT_ROOT)

ANTIBIOTIC_ANALYSIS_LEVEL = cfg(PIPELINE_CONFIG, "glass_interactive.antibiotic_analysis_level")
SI_PATHS = build_si_analysis_paths(PROJECT_ROOT, ANTIBIOTIC_ANALYSIS_LEVEL)
YEARLY_SCREENED_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
YEARLY_SI_PATH = SI_PATHS["yearly_si"]
RANK_OUTPUT_PATH = SI_PATHS["rank_output"]
INTERESTING_OUTPUT_PATH = SI_PATHS["interesting_output"]

GLASS_XLSX_PATH = cfg_path(PIPELINE_CONFIG, "paths.glass_xlsx", PROJECT_ROOT)
GLASS_CLEANED_PATH = cfg_path(PIPELINE_CONFIG, "paths.glass_cleaned", PROJECT_ROOT)
BACTERIA_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogens", PROJECT_ROOT)
ANTIBIOTIC_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_synonyms", PROJECT_ROOT)
ANTIBIOTIC_COARSENING_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_coarsening", PROJECT_ROOT)

# Reuse SI fit window settings so interactive plots stay aligned with notebook 05 outputs.
FIT_START_YEAR = cfg(PIPELINE_CONFIG, "si.fit_start_year")
FIT_END_YEAR = cfg(PIPELINE_CONFIG, "si.fit_end_year")
CURVE_FIT_END_YEAR = cfg(PIPELINE_CONFIG, "si.curve_fit_end_year")
NUM_FUTURE_YEARS = cfg(PIPELINE_CONFIG, "glass_interactive.num_future_years")
LOESS_FRAC = cfg(PIPELINE_CONFIG, "si.loess_frac")
LOESS_DEGREE = cfg(PIPELINE_CONFIG, "si.loess_degree")
TAIL_DAMPING_YEARS = cfg(PIPELINE_CONFIG, "si.tail_damping_years")
MIN_AMR_PAIR_PUBLICATION_COUNT = cfg(PIPELINE_CONFIG, "si.min_amr_pair_publication_count")
MIN_NUM_YEARS_WITH_VALID_COUNTS = cfg(PIPELINE_CONFIG, "si.min_num_years_with_valid_counts")

# Interactive filtering and plot layout controls.
MIN_LAST_AVAILABLE_COUNT = cfg(PIPELINE_CONFIG, "glass_interactive.min_last_available_count")
MIN_TOTAL_PUBLICATIONS = cfg(PIPELINE_CONFIG, "glass_interactive.min_total_publications")
MIN_COUNT_ABS_PCT_ERROR_MEAN = cfg(PIPELINE_CONFIG, "glass_interactive.min_count_abs_pct_error_mean")
MAX_COUNT_ABS_PCT_ERROR_MEAN = cfg(PIPELINE_CONFIG, "glass_interactive.max_count_abs_pct_error_mean")
MIN_COUNTS_BACKGROUND = cfg(PIPELINE_CONFIG, "glass_interactive.min_counts_background")
MIN_ABS_DELTA_SI = cfg(PIPELINE_CONFIG, "glass_interactive.min_abs_delta_si")
MIN_SI_DELTA = cfg(PIPELINE_CONFIG, "glass_interactive.min_si_delta")
DISPLAY_NAME = cfg(PIPELINE_CONFIG, "glass_interactive.display_name")
MAX_TEXT_LABELS = cfg(PIPELINE_CONFIG, "glass_interactive.max_text_labels")
WIDTH = cfg(PIPELINE_CONFIG, "glass_interactive.width")
HEIGHT = cfg(PIPELINE_CONFIG, "glass_interactive.height")

if not YEARLY_SCREENED_PATH.exists():
    raise FileNotFoundError(f"Missing {YEARLY_SCREENED_PATH}. Run notebook 04 to create screened AMR pairs first.")
if not YEARLY_SI_PATH.exists():
    raise FileNotFoundError(f"Missing {YEARLY_SI_PATH}. Run notebook 05 after notebook 04 to build SI outputs.")
if YEARLY_SI_PATH.stat().st_mtime < YEARLY_SCREENED_PATH.stat().st_mtime:
    raise RuntimeError(
        f"{YEARLY_SI_PATH} is older than {YEARLY_SCREENED_PATH}. "
        "Run notebook 05 after the latest notebook 04 LLM/screened-pair run."
    )


## Load SI And Clean GLASS List

In [4]:
si_df, glass_pairs_df = load_si_and_clean_glass_list(
    YEARLY_SI_PATH,
    GLASS_XLSX_PATH,
    bacteria_synonym_path=BACTERIA_SYNONYM_PATH,
    antibiotic_synonym_path=ANTIBIOTIC_SYNONYM_PATH,
    antibiotic_coarsening_path=ANTIBIOTIC_COARSENING_PATH,
    glass_cleaned_path=GLASS_CLEANED_PATH,
)

si_source_df = pd.read_parquet(YEARLY_SCREENED_PATH)
si_pathogen_summary_df = summarize_pathogen_coverage(
    source_df=si_source_df,
    modeled_df=si_df,
    source_label="screened AMR input from notebook 04",
    modeled_label="filtered SI dataset from notebook 05",
)
display(si_pathogen_summary_df.query("status != 'present_in_both'"))
display(glass_pairs_df.head(20))

Loaded SI rows: 15,806 from /Users/f.costa/Code/Antimicrobial-Resistance/outputs/parquets/yearly_si.parquet
Pathogens: 58; antibiotics/levels: 96
Saved cleaned GLASS rows: 96 to /Users/f.costa/Code/Antimicrobial-Resistance/assets/glass_amr_cleaned.csv
screened AMR input from notebook 04 pathogens: 719
filtered SI dataset from notebook 05 pathogens: 58
Pathogens present in screened AMR input from notebook 04 but absent from filtered SI dataset from notebook 05: 661
Abiotrophia defectiva, Achromobacter denitrificans, Achromobacter mucicolens, Achromobacter xylosoxidans, Acinetobacter bereziniae, Acinetobacter calcoaceticus, Acinetobacter haemolyticus, Acinetobacter johnsonii, Acinetobacter junii, Acinetobacter lwoffii, Acinetobacter nosocomialis, Acinetobacter pittii, Acinetobacter radioresistens, Acinetobacter schindleri, Acinetobacter seifertii, Acinetobacter soli, Acinetobacter ursingii, Actinobacillus equuli, Actinobacillus hominis, Actinobacillus ureae, Actinomadura madurae, Actinom

,pathogen,source_rows,source_publications,source_pairs,modeled_rows,modeled_publications,modeled_pairs,in_source,in_modeled,status
0,Abiotrophia defectiva,50,53,23,0.0,0.0,0.0,True,False,missing_from_modeled
1,Achromobacter denitrificans,10,10,7,0.0,0.0,0.0,True,False,missing_from_modeled
2,Achromobacter mucicolens,16,16,15,0.0,0.0,0.0,True,False,missing_from_modeled
3,Achromobacter xylosoxidans,270,382,60,0.0,0.0,0.0,True,False,missing_from_modeled
4,Acinetobacter bereziniae,19,19,12,0.0,0.0,0.0,True,False,missing_from_modeled
...,...,...,...,...,...,...,...,...,...,...
656,Winkia neuii,11,11,11,0.0,0.0,0.0,True,False,missing_from_modeled
657,Wohlfahrtiimonas chitiniclastica,17,17,13,0.0,0.0,0.0,True,False,missing_from_modeled
658,Yersinia pestis,85,106,30,0.0,0.0,0.0,True,False,missing_from_modeled
659,Yersinia pseudotuberculosis,87,89,33,0.0,0.0,0.0,True,False,missing_from_modeled


,raw_pathogen,raw_antibiotic,pathogen,antibiotic,pathogen_standardized,antibiotic_standardized,Class,Simplified categories,High level categories,Origin
0,Acinetobacter spp.,amikacin,Acinetobacter baumannii,Amikacin,True,True,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
1,Acinetobacter spp.,Colistin,Acinetobacter baumannii,Colistin,True,False,Polymyxins,Miscellaneous,Other,Naturally occurring/modified
2,Acinetobacter spp.,doripenem,Acinetobacter baumannii,Doripenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
3,Acinetobacter spp.,Gentamicin,Acinetobacter baumannii,Gentamicin,True,False,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
4,Acinetobacter spp.,Imipenem,Acinetobacter baumannii,Imipenem/cilastatin,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
5,Acinetobacter spp.,meropenem,Acinetobacter baumannii,Meropenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
6,Acinetobacter spp.,minocycline,Acinetobacter baumannii,Minocycline,True,True,Tetracyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
7,Acinetobacter spp.,Tigecycline,Acinetobacter baumannii,Tigecycline,True,False,Glycylcyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
8,E. coli,Cefepime,Escherichia coli,Cefepime,True,False,Fourth-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified
9,E. coli,cefotaxime,Escherichia coli,Cefotaxime,True,True,Third-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified


## Build Or Load Rank Data

In [5]:
rank_df = load_or_build_interactive_rank(
    si_df,
    RANK_OUTPUT_PATH,
    source_path=YEARLY_SI_PATH,
    fit_start_year=FIT_START_YEAR,
    fit_end_year=FIT_END_YEAR,
    curve_fit_end_year=CURVE_FIT_END_YEAR,
    num_future_years=NUM_FUTURE_YEARS,
    loess_frac=LOESS_FRAC,
    loess_degree=LOESS_DEGREE,
    tail_damping_years=TAIL_DAMPING_YEARS,
    min_amr_pair_publication_count=MIN_AMR_PAIR_PUBLICATION_COUNT,
    min_num_years_with_valid_counts=MIN_NUM_YEARS_WITH_VALID_COUNTS,
    verbose=True,
)

if not INTERESTING_OUTPUT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {INTERESTING_OUTPUT_PATH}. Run notebook 05 first so only pairs satisfying all SI criteria enter the GLASS analysis."
    )
if INTERESTING_OUTPUT_PATH.stat().st_mtime < YEARLY_SI_PATH.stat().st_mtime:
    raise RuntimeError(
        f"{INTERESTING_OUTPUT_PATH} is older than {YEARLY_SI_PATH}. Rerun notebook 05 so its configured SI eligibility criteria are applied."
    )
eligible_pairs = pd.read_parquet(INTERESTING_OUTPUT_PATH)[["pathogen", "antibiotic"]].drop_duplicates()
rank_df = rank_df.merge(eligible_pairs, on=["pathogen", "antibiotic"], how="inner", validate="one_to_one")
print(f"Eligible SI pairs entering Notebook 06: {len(rank_df):,}")
display(rank_df)

Recomputing rank data because /Users/f.costa/Code/Antimicrobial-Resistance/outputs/parquets/yearly_si_ranked_future.parquet is older than /Users/f.costa/Code/Antimicrobial-Resistance/outputs/parquets/yearly_si.parquet
[1/644] ranking Acinetobacter baumannii vs Amikacin
[2/644] ranking Acinetobacter baumannii vs Ampicillin
[3/644] ranking Acinetobacter baumannii vs Ampicillin/sulbactam
[4/644] ranking Acinetobacter baumannii vs Aztreonam
[5/644] ranking Acinetobacter baumannii vs Cefepime
[6/644] ranking Acinetobacter baumannii vs Cefiderocol
[7/644] ranking Acinetobacter baumannii vs Cefoperazone
[8/644] ranking Acinetobacter baumannii vs Cefotaxime
[9/644] ranking Acinetobacter baumannii vs Ceftazidime
[10/644] ranking Acinetobacter baumannii vs Ceftazidime/avibactam
[11/644] ranking Acinetobacter baumannii vs Ceftriaxone
[12/644] ranking Acinetobacter baumannii vs Chloramphenicol
[13/644] ranking Acinetobacter baumannii vs Ciprofloxacin
[14/644] ranking Acinetobacter baumannii vs Col

,pathogen,antibiotic,reference_year,future_year,num_future_years,fit_start_year,fit_end_year,curve_fit_end_year,post_fit_validation_end_year,min_amr_pair_publication_count,...,tail_damping_years,validation_loess_frac,validation_loess_degree,validation_tail_damping_years,status,error,current_rank,future_rank,rank_change,abs_rank_change
0,Mycobacterium tuberculosis,Linezolid,2025,2028,3,2000,2023,2025,2025,3.0,...,NaN,0.6,1,NaN,ok,NaN,399.0,305.0,94.0,94.0
1,Escherichia coli,Ceftazidime/avibactam,2025,2028,3,2000,2023,2025,2025,3.0,...,NaN,0.6,1,NaN,ok,NaN,293.0,200.0,93.0,93.0
2,Acinetobacter baumannii,Sulfamethoxazole,2025,2028,3,2000,2023,2025,2025,3.0,...,NaN,0.6,1,NaN,ok,NaN,302.0,227.0,75.0,75.0
3,Mycobacterium abscessus,Cefoxitin,2025,2028,3,2000,2023,2025,2025,3.0,...,NaN,0.6,1,NaN,ok,NaN,107.0,36.0,71.0,71.0
4,Escherichia coli,Cefixime,2025,2028,3,2000,2023,2025,2025,3.0,...,NaN,0.6,1,NaN,ok,NaN,165.0,99.0,66.0,66.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,Escherichia coli,Ampicillin,2025,2028,3,2000,2023,2025,2025,3.0,...,NaN,0.6,1,NaN,ok,NaN,2.0,2.0,0.0,0.0
123,Escherichia coli,Sulfamethoxazole,2025,2028,3,2000,2023,2025,2025,3.0,...,NaN,0.6,1,NaN,ok,NaN,32.0,32.0,0.0,0.0
124,Pseudomonas aeruginosa,Colistin,2025,2028,3,2000,2023,2025,2025,3.0,...,NaN,0.6,1,NaN,ok,NaN,66.0,66.0,0.0,0.0
125,Stenotrophomonas maltophilia,Trimethoprim,2025,2028,3,2000,2023,2025,2025,3.0,...,NaN,0.6,1,NaN,ok,NaN,86.0,86.0,0.0,0.0


## Interactive Latest Count Vs Future SI Shift

In [6]:
latest_count_shift_result = plot_interactive_latest_observed_count_vs_future_si_shift(
    rank_df,
    glass_pairs_df=glass_pairs_df,
    min_last_available_count=MIN_LAST_AVAILABLE_COUNT,
    min_total_publications=MIN_TOTAL_PUBLICATIONS,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    min_counts_background=MIN_COUNTS_BACKGROUND,
    min_abs_delta_si=MIN_ABS_DELTA_SI,
    min_si_delta=MIN_SI_DELTA,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "interactive_latest_observed_count_vs_future_si_shift.pdf",
    show=True,
)
latest_count_shift_df = latest_count_shift_result["plot_df"]
print(f"Displayed rows: {len(latest_count_shift_df):,}; GLASS matches: {int(latest_count_shift_df['is_glass_pair'].sum()):,}")
if latest_count_shift_result["pdf_path"]:
    print(f"Saved PDF: {latest_count_shift_result['pdf_path']}")
elif latest_count_shift_result.get("pdf_error"):
    print(f"PDF export failed: {latest_count_shift_result['pdf_error']}")

Displayed rows: 123; GLASS matches: 31
Saved PDF: /Users/f.costa/Code/Antimicrobial-Resistance/outputs/figures/interactive_latest_observed_count_vs_future_si_shift.pdf


## Display GLASS Matches In Plot

In [7]:
glass_plot_matches_df = build_glass_match_table(latest_count_shift_df)
display(glass_plot_matches_df)

,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,total_publications,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean,glass_group_column
0,Escherichia coli,Ciprofloxacin,2025.0,2028,266.0,3589.0,292.564235,0.677392,0.714530,0.037138,9.713938,antibiotic
1,Klebsiella pneumoniae,Ceftazidime,2025.0,2028,188.0,1754.0,353.731821,0.680547,0.818079,0.137531,15.726750,antibiotic
2,Klebsiella pneumoniae,Meropenem,2025.0,2028,185.0,1957.0,286.402388,0.689531,0.743919,0.054388,20.622386,antibiotic
3,Klebsiella pneumoniae,Colistin,2025.0,2028,176.0,1958.0,219.321974,0.463271,0.471653,0.008381,23.865680,antibiotic
4,Pseudomonas aeruginosa,Ceftazidime,2025.0,2028,159.0,2141.0,200.147834,0.598209,0.631699,0.033489,14.838768,antibiotic
5,Escherichia coli,Meropenem,2025.0,2028,158.0,1635.0,265.522775,0.522091,0.626981,0.104890,25.445255,antibiotic
6,Pseudomonas aeruginosa,Meropenem,2025.0,2028,154.0,1955.0,202.856863,0.614727,0.654759,0.040032,5.133357,antibiotic
7,Pseudomonas aeruginosa,Colistin,2025.0,2028,138.0,1477.0,165.440150,0.338089,0.384189,0.046100,3.014770,antibiotic
8,Escherichia coli,Sulfamethoxazole/trimethoprim,2025.0,2028,125.0,1882.0,168.913585,0.521364,0.561814,0.040450,10.599733,antibiotic
9,Escherichia coli,Ceftriaxone,2025.0,2028,125.0,1324.0,160.027664,0.486593,0.571176,0.084583,22.090488,antibiotic


## Novel Non-GLASS Predictions

These are plotted pairs that do not appear in the cleaned GLASS list, sorted as candidate novel predictions of interest.

In [8]:
novel_prediction_df = build_novel_prediction_table(latest_count_shift_df)
print(f"Novel non-GLASS predictions displayed: {len(novel_prediction_df):,}")
display(novel_prediction_df)

Novel non-GLASS predictions displayed: 92


,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,total_publications,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean
0,Klebsiella pneumoniae,Amoxicillin/clavulanic-acid,2025.0,2028,35.0,208.0,65.056823,0.286213,0.509119,0.222906,29.245519
1,Escherichia coli,Cefixime,2025.0,2028,24.0,184.0,44.396063,0.121082,0.308546,0.187464,23.196526
2,Escherichia coli,Vancomycin,2025.0,2028,16.0,116.0,21.902709,-0.811414,-0.634878,0.176535,29.679495
3,Klebsiella pneumoniae,Nitrofurantoin,2025.0,2028,29.0,210.0,48.941851,0.195392,0.357093,0.161701,4.147175
4,Escherichia coli,Ceftazidime/avibactam,2025.0,2028,45.0,288.0,72.313038,-0.081806,0.071598,0.153405,15.788186
...,...,...,...,...,...,...,...,...,...,...,...
87,Staphylococcus aureus,Erythromycin,2025.0,2028,122.0,2013.0,128.573746,0.567654,0.578430,0.010776,6.932503
88,Stenotrophomonas maltophilia,Levofloxacin,2025.0,2028,18.0,235.0,22.151175,0.262067,0.271792,0.009724,36.512455
89,Mycobacterium tuberculosis,Rifampicin,2025.0,2028,215.0,3464.0,212.271965,0.981312,0.990512,0.009200,7.954191
90,Escherichia coli,Sulbactam,2025.0,2028,28.0,386.0,29.114897,-0.000002,0.009112,0.009114,18.424087


## Future SI Vs Future SI Shift

This uses the same y-axis definition as the latest-count plot: predicted future SI minus smoothed SI at the latest available year.

In [12]:
future_si_shift_result = plot_future_si_vs_si_shift(
    latest_count_shift_df,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "future_si_vs_current_si_quadrant.pdf",
    show=True,
)
future_si_shift_fig = future_si_shift_result["figure"]
if future_si_shift_result["pdf_path"]:
    print(f"Saved PDF: {future_si_shift_result['pdf_path']}")
elif future_si_shift_result.get("pdf_error"):
    print(f"PDF export failed: {future_si_shift_result['pdf_error']}")

Saved PDF: /Users/f.costa/Code/Antimicrobial-Resistance/outputs/figures/future_si_vs_current_si_quadrant.pdf


In [11]:
latest_count_shift_df.to_csv(
    FIGURE_OUTPUT_DIR / "future_si_vs_current_si_quadrant.csv",
    index=False,
)